In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

chinese_fonts = ['Microsoft YaHei', 'SimHei', 'STHeiti', 'SimSun', 'Arial Unicode MS']
sns.set_theme(style="white", rc={"font.sans-serif": chinese_fonts})
plt.rcParams['font.sans-serif'] = chinese_fonts
plt.rcParams['axes.unicode_minus'] = False

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 ADS 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.ads_top10_attention_houses_daily 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_historical_hot_houses_area(city_name):
    """
    获取指定城市高关注房源面积数据。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        house_id,
        MAX(area_sqm) as area_sqm
    FROM iceberg_catalog.ershoufang.ads_top10_attention_houses_daily
    WHERE city = '{city_name}' 
      AND area_sqm > 0 
    GROUP BY house_id
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_area_violin_scatter(city):
    """绘制小提琴图与散点叠加图"""
    df = fetch_historical_hot_houses_area(city)
    
    if df.empty or len(df) < 10:
        plt.figure(figsize=(12, 6))
        plt.text(0.5, 0.5, f"{city} 历史高关注房源样本过少，无法绘制面积分布图", 
                 ha='center', va='center', fontsize=14, color='#7F8C8D')
        plt.axis('off')
        plt.show()
        return

    # --- 1. 数据清洗 ---
    p99_area = df['area_sqm'].quantile(0.99)
    df_clean = df[df['area_sqm'] <= p99_area].copy()
    df_clean['category'] = '全市热门上榜房源'

    # --- 2. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.violinplot(
        data=df_clean, 
        x='area_sqm', 
        y='category', 
        color='#AED6F1',   
        inner='quartile', 
        linewidth=1.5, 
        linecolor='#2980B9',
        ax=ax
    )
    sns.stripplot(
        data=df_clean, 
        x='area_sqm', 
        y='category', 
        color='#154360',      
        size=4, 
        alpha=0.5,            
        jitter=0.2, 
        ax=ax
    )

    # --- 3. 标注与修饰 ---
    median_area = df_clean['area_sqm'].median()
    ax.axvline(median_area, color='#E74C3C', linestyle='-', linewidth=2, alpha=0.8, zorder=5)
    ax.text(
        median_area + (p99_area * 0.01), 
        -0.35, 
        f'中位数面积: {median_area:.1f} ㎡', 
        color='#E74C3C', fontweight='bold', va='center', fontsize=11
    )

    plt.title(f'[{city}] 高关注度二手房面积分布画像 (小提琴密度 + 颗粒散点)', fontsize=16, pad=20, fontweight='bold')
    plt.xlabel('房源建筑面积 (平方米)', fontsize=12, labelpad=10)
    
    # 隐藏 Y 轴的 label 文本，仅保留分类名称
    plt.ylabel('')
    ax.tick_params(axis='y', labelsize=12)

    # 移除顶部、右侧、左侧的边框线
    sns.despine(left=True, top=True, right=True)
    # 添加极简的 X 轴垂直网格线
    ax.grid(axis='x', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        print("未获取到城市列表，请检查数据。")
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_area_violin_scatter(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_area_violin_scatter(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=1, layout=Layout(width='250px'), options=('上海', '北京', '南京', '天津', '太原', …

Output()